#### Create events file from HCP behavioral data saved as psychopy outputs

In [1]:
import pandas as pd
import numpy as np
import os.path as op

In [ ]:
home_dir  = "/home/jovyan"
participant = "HCA6002236_V3_MR"
data_dir = op.join(home_dir, "shared", "data", "HCP-Aging", participant, "unprocessed/tfMRI_VISMOTOR_PA/LINKED_DATA/PSYCHOPY")
out_path = op.join(home_dir, "behavioral_visuomotor_task_events")

In [17]:
fname = f"VISMOTOR_{participant}_A_run1_wide.csv"
fname_corr = fname.replace("_MR", "").rstrip("_")

psychopy_file = op.join(data_dir, fname_corr)
op.exists(data_dir)

df = pd.read_csv(psychopy_file)

# Keep only actual task trials
trials = df[df["targetStartTime"].notna() & df["targetEndTime"].notna()].copy()

# Create BIDS/Nilearn-style events table
events = pd.DataFrame({
    "onset": trials["targetStartTime"].astype(float),
    "duration": (trials["targetEndTime"] - trials["targetStartTime"]).astype(float),
    "trial_type": trials["side"].astype(str)
})

# Optional: add accuracy / response info as extra columns
events["accuracy"] = trials["correct"].astype(str)
events["rt"] = pd.to_numeric(trials["firstRt"], errors="coerce")

# Save in a format Nilearn can read
events.to_csv(op.join(out_path, "sub-01_task-vismotor_run-01_events.tsv"), sep="\t", index=False)
print(events.head())

       onset  duration trial_type accuracy        rt
4  28.067597  0.500217      right     True  0.469688
5  31.066849  0.500953       left     True  0.485486
6  34.051472  0.499275      right     True  0.501869
7  37.067954  0.500010       left     True  0.519505
8  40.051857  0.499080      right     True  0.436894
